# Shape optimization

This is the actual point of the whole project: using the trained network not just to
predict Cd, but to *improve* a shape.

The idea: freeze every weight in the model. Instead of updating the network's
parameters with gradient descent (normal training), update the input mesh's own
vertex coordinates instead. A forward pass gives a predicted Cd; backpropagation
sends gradients from that single number all the way back through the frozen network
into the raw (x, y, z) positions of the mesh. Taking small steps against that
gradient reshapes the car toward lower predicted drag — the same math as training,
just pointed at the input instead of the weights.

Done naively, this doesn't work well: an unconstrained optimizer finds tiny,
physically meaningless mesh distortions that fool the model into predicting an
unrealistic improvement, because the Cd head is trained on very little data (see the
training notebook) and isn't a fully trustworthy objective to chase indefinitely.
Most of this notebook is about controlling for that and then checking the result
against a second, more trustworthy signal — not just running the optimizer and
believing whatever comes out.


Load the trained checkpoint and freeze it — `model.eval()` plus, later, turning off
`requires_grad` on every parameter inside `optimize_shape`, so gradients only flow
into the mesh coordinates, never back into the network's weights.


In [1]:
import sys, json
sys.path.insert(0, "..")

import torch
from src.model import MeshSurrogate
from src.shape_opt import optimize_shape

device = "cuda" if torch.cuda.is_available() else "cpu"

with open("../outputs/norm_stats.json") as f:
    stats = json.load(f)

model = MeshSurrogate(in_channels=6).to(device)
model.load_state_dict(torch.load("../outputs/checkpoints/best_model.pt", weights_only=True, map_location=device))
model.eval()

test_data = torch.load("../outputs/cache/test_pyg.pt", weights_only=False)
print(len(test_data), "test samples available")


111 test samples available


Pick a held-out test car to optimize. `pos_init` is already normalized the same
way the model was trained on — this matters, since optimizing in the wrong
coordinate scale would make the regularizer weights below meaningless.

Car `753` specifically: with the safeguards below in place, the resulting geometry
change is small on *every* test shape (that's the honest, expected outcome given how
little Cd training data there is — see the README). To make that small change as
visible as possible rather than picking an arbitrary car, all 111 test shapes were
scanned for the one with the largest vertex displacement after optimization; `753`
came out on top, with the largest legitimate (pressure-verified) improvement too.


In [2]:
from src.data import load_mesh

sample = next(s for s in test_data if s.sample_id == "753")
sample_id = sample.sample_id
_, faces = load_mesh(sample_id)
pos_init = sample.x[:, :3].clone()
faces_t = torch.from_numpy(faces).long()
edge_index_t = sample.edge_index

print(sample_id, pos_init.shape, faces_t.shape)

753 torch.Size([3586, 3]) torch.Size([7168, 3])


Run the optimization. Three choices here carry all the weight:

**The variables are FFD control points, not vertices.** Descending directly on the
3586 vertex positions gives the optimizer 10758 degrees of freedom against a
surrogate trained on 450 shapes, so almost every descent direction leaves the space
of plausible car geometry -- the earlier version of this notebook did exactly that
and produced a lumpy, adversarial mesh whose "improvement" was the Cd head being
fooled. A 4x4x6 Bernstein cage (`src/ffd.py`) expresses the deformation as a smooth
polynomial over 288 parameters instead. High-frequency surface noise is not
penalized after the fact; it simply cannot be represented. This also makes the old
`alpha_smooth` Laplacian term and the gradient-smoothing pass unnecessary, and both
have been removed.

**The objective is drag force, with frontal area and volume pinned.** Cd and drag
force are gameable in opposite directions: frontal area sits in Cd's denominator, so
a wider car scores a lower Cd, while a smaller car trivially has less drag force.
`gamma_area` and `gamma_volume` hold both to within about 0.1% of their starting
values, which leaves reshaping as the only way to improve and makes the two
objectives equivalent.

**`CD_PHYSICAL_SIGN`** corrects the model's raw output to the sign that actually
corresponds to real drag. Getting this wrong makes the optimizer improve the wrong
thing with no error and no obviously bad-looking output, which is what happened
during development before the sign was derived (see the README).

In [3]:
from src.shape_opt import CD_PHYSICAL_SIGN

# Optimizes FFD control points (288 DOF) rather than raw vertices (10758), against
# predicted drag *force* with frontal area and volume pinned -- see src/shape_opt.py
# for why every one of those choices is load-bearing.
history = optimize_shape(
    model, pos_init, faces_t, edge_index_t,
    n_steps=300, lr=2e-3, beta_anchor=20.0,
    gamma_area=300.0, gamma_volume=300.0,
    max_rel_cd_change=0.10, lattice_dims=(4, 4, 6),
    cd_mean=stats["cd_mean"], cd_std=stats["cd_std"],
    pressure_mean=stats["pressure_mean"], pressure_std=stats["pressure_std"], device=device,
)

cd_std, cd_mean = stats["cd_std"], stats["cd_mean"]
cd_over_steps = [CD_PHYSICAL_SIGN * (h["cd_pred"] * cd_std + cd_mean) for h in history]
drag_over_steps = [h["drag_force"] for h in history]
print(f"Cd (physical sign): start {cd_over_steps[0]:.2f}, end {cd_over_steps[-1]:.2f}")
print(f"drag force:         start {drag_over_steps[0]:.2f}, end {drag_over_steps[-1]:.2f}")
print(f"{len(history)} steps run before hitting the 10% early-stop threshold")

Cd (physical sign): start 61.51, end 55.09
drag force:         start 29.99, end 26.85
18 steps run before hitting the 10% early-stop threshold


## Does an independent signal agree with the Cd head?

The Cd head is fit on only ~450 single-scalar labels and visibly overfits (see
`train_model.ipynb`'s train/val gap), so a decrease in its own output is weak
evidence by itself. Instead, recompute Cd from the model's **pressure** prediction
(the much better-supervised head, trained on ~3600 targets/shape) using the exact
same analytic pressure-drag integral used to build the original training labels.
If that independently-derived number also drops for the optimized shape, the two
heads agree — real evidence of a self-consistent effect, not just one head being
fooled. Plain geometric facts (surface/frontal area, vertex displacement) are also
reported so the shapes can be compared without trusting either head at all.


In [4]:
from src.shape_opt import verify_optimization

verification = verify_optimization(
    model, pos_init, history[-1]["pos"], faces_t, edge_index_t,
    pressure_mean=stats["pressure_mean"], pressure_std=stats["pressure_std"],
    cd_mean=stats["cd_mean"], cd_std=stats["cd_std"], device=device,
)

print(f"Cd head:            {verification['before']['cd_head_raw']:.2f} -> {verification['after']['cd_head_raw']:.2f}  ({verification['cd_head_pct_change']:+.1f}%)")
print(f"Drag force:         {verification['before']['analytic_drag_force']:.2f} -> {verification['after']['analytic_drag_force']:.2f}  ({verification['drag_force_pct_change']:+.1f}%)")
print(f"Analytic Cd (fixed reference area): {verification['analytic_pct_change']:+.1f}%")
print()
print("Package constraints -- these must stay near zero, or the 'improvement' is just resizing:")
print(f"  Frontal area: {verification['before']['frontal_area']:.4f} -> {verification['after']['frontal_area']:.4f}  ({verification['frontal_area_pct_change']:+.2f}%)")
print(f"  Volume:       {verification['before']['volume']:.4f} -> {verification['after']['volume']:.4f}  ({verification['volume_pct_change']:+.2f}%)")
print(f"  Surface area: {verification['before']['surface_area']:.4f} -> {verification['after']['surface_area']:.4f}")
print()
print(f"Vertex displacement: max={verification['displacement']['max']:.4f}, mean={verification['displacement']['mean']:.4f}  (mesh spans ~2.0 units)")

with open("../outputs/shape_opt_verification.json", "w") as f:
    json.dump(verification, f, indent=2)

Cd head:            61.51 -> 55.09  (-10.4%)
Drag force:         34.10 -> 30.23  (-11.3%)
Analytic Cd (fixed reference area): -11.3%

Package constraints -- these must stay near zero, or the 'improvement' is just resizing:
  Frontal area: 0.4875 -> 0.4874  (-0.03%)
  Volume:       0.5910 -> 0.5925  (+0.26%)
  Surface area: 4.9503 -> 4.9338

Vertex displacement: max=0.0335, mean=0.0129  (mesh spans ~2.0 units)


### Repeat across multiple held-out shapes

One shape is an anecdote. Run the same optimize-then-verify procedure on several
held-out test cars and report how often the independent (pressure-integral) signal
agrees with the Cd head's claimed direction of improvement.


In [5]:
N_SHAPES = 10
multi_results = []

for i in range(N_SHAPES):
    s = test_data[i]
    _, f = load_mesh(s.sample_id)
    f_t = torch.from_numpy(f).long()
    p0 = s.x[:, :3].clone()

    hist = optimize_shape(
        model, p0, f_t, s.edge_index,
        n_steps=300, lr=2e-3, beta_anchor=20.0,
        gamma_area=300.0, gamma_volume=300.0, max_rel_cd_change=0.10,
        cd_mean=stats["cd_mean"], cd_std=stats["cd_std"],
        pressure_mean=stats["pressure_mean"], pressure_std=stats["pressure_std"], device=device,
    )
    v = verify_optimization(
        model, p0, hist[-1]["pos"], f_t, s.edge_index,
        pressure_mean=stats["pressure_mean"], pressure_std=stats["pressure_std"],
        cd_mean=stats["cd_mean"], cd_std=stats["cd_std"], device=device,
    )
    v["sample_id"] = s.sample_id
    v["n_steps_run"] = len(hist)
    multi_results.append(v)
    print(f"{s.sample_id}: drag force {v['drag_force_pct_change']:+.2f}%  cd_head {v['cd_head_pct_change']:+.2f}%  "
          f"area {v['frontal_area_pct_change']:+.2f}%  vol {v['volume_pct_change']:+.2f}%  steps={len(hist)}")

improved = sum(r["drag_force_pct_change"] < 0 for r in multi_results)
print(f"\nDrag force reduced on {improved}/{N_SHAPES} held-out shapes")
print("Compare each change against the Cd head's test RMSE before treating it as real.")

with open("../outputs/shape_opt_verification_multi.json", "w") as f:
    json.dump(multi_results, f, indent=2)

658: drag force -9.08%  cd_head -10.79%  area +0.11%  vol +0.19%  steps=12


659: drag force -11.16%  cd_head -9.97%  area -0.36%  vol -0.07%  steps=20


660: drag force -14.98%  cd_head -10.46%  area +0.11%  vol +0.14%  steps=19


662: drag force -12.18%  cd_head -10.20%  area -0.34%  vol -0.42%  steps=16


663: drag force -10.48%  cd_head -10.25%  area -0.14%  vol -0.18%  steps=15


664: drag force -13.79%  cd_head -10.19%  area -0.27%  vol -0.25%  steps=15


665: drag force -8.76%  cd_head -9.96%  area -0.11%  vol -0.01%  steps=22


666: drag force -11.27%  cd_head -10.50%  area -0.01%  vol +0.25%  steps=18


667: drag force -9.91%  cd_head -10.29%  area -0.17%  vol -0.18%  steps=15


668: drag force -7.64%  cd_head -10.05%  area -0.06%  vol +0.19%  steps=18

Drag force reduced on 10/10 held-out shapes
Compare each change against the Cd head's test RMSE before treating it as real.


## Saving the result

Save the single-car trajectory (positions, predicted pressure, and Cd at every step)
for the visualization notebook to turn into figures and animations.


In [6]:
import torch as _torch
_torch.save(
    {
        "sample_id": sample_id, "faces": faces, "cd_over_steps": cd_over_steps,
        "pos_history": [h["pos"].numpy() for h in history],
        "pressure_history": [h["pressure"] for h in history],
        "verification": verification,
    },
    "../outputs/shape_opt_result.pt",
)
